In [ ]:
import numpy as np
import pandas as pd
import os
import datetime

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import KFold, StratifiedKFold
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader,Subset, random_split
import torch.nn.functional as F

import time
import pennylane as qml

from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
import plotly.express as px

from pytorch_model_summary import summary
import pathlib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_path = 'data/binary_brain_tumors_detection'
class_paths  = os.listdir(data_path + '/classes')
print(class_paths)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.ImageFolder(data_path + '/classes', transform=transform)

lengths = [int(np.ceil(0.7*len(dataset))),
           int(np.floor(0.15*len(dataset))),
           int(np.floor(0.15*len(dataset)))]
train_set, test_set, val_set = random_split(dataset, lengths)

train_dataloader = DataLoader(train_set, batch_size=100)
test_dataloader = DataLoader(test_set, batch_size=100)
val_dataloader = DataLoader(val_set, batch_size=100)


device = torch.device("cpu") # torch.device("cuda" if torch.cuda.is_available() else "cpu")

print('lengths',lengths)

In [ ]:
# To check memory usage
import psutil

# inner psutil function
def process_memory():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss

# decorator function
def profile(func):
    def wrapper(*args, **kwargs):

        mem_before = process_memory()
        result = func(*args, **kwargs)
        mem_after = process_memory()
        print("{}:consumed memory: {:,}".format(
            func.__name__,
            mem_before, mem_after, mem_after - mem_before))

        return result

In [ ]:
@qml.qnode(dev, interface="torch")
def qnode(inputs, weights): 
    for i in range(n_base_chain):
        qml.AngleEmbedding(inputs, wires=range(n_qubits))
        qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(wires=i)) for i in range(2)]
    
weight_shapes = {"weights": (n_layers, n_qubits)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

In [ ]:
class QuantBrainTumorConvNet(nn.Module):

    def __init__(self):
        super().__init__()

        # onvolutional layers (3,16,32)
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size=(5, 5), stride=2, padding=1)
        self.conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size=(5, 5), stride=2, padding=1)
        self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=(3, 3), padding=1)

        # conected layers
        self.fc1 = nn.Linear(in_features= 64 * 6 * 6, out_features=500)
        self.fc2 = nn.Linear(in_features=500, out_features=50)
        self.fc3 = nn.Linear(in_features=50, out_features=n_qubits)
        # last layer as a quantum circuit
        self.quantum_layer = qlayer
        self.fc4 = nn.Linear(in_features=n_qubits, out_features=2)


    def forward(self, X):
        
        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, 2)

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, 2)

        X = F.relu(self.conv3(X))
        X = F.max_pool2d(X, 2)

        X = X.view(X.shape[0], -1)
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        #print('1 ',X.shape)
        #print(X)
        X = self.quantum_layer(X)
        #print('2 ',X.shape)

        return X

In [ ]:
class ClassicBrainTumorConvNet(nn.Module):

    def __init__(self):
        super().__init__()

        # сonvolutional layers (3,16,32)
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 16, kernel_size=(5, 5), stride=2, padding=1)
        self.conv2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size=(5, 5), stride=2, padding=1)
        self.conv3 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size=(3, 3), padding=1)

        # conected layers
        self.fc1 = nn.Linear(in_features= 64 * 6 * 6, out_features=500)
        self.fc2 = nn.Linear(in_features=500, out_features=50)
        self.fc3 = nn.Linear(in_features=50, out_features=n_qubits)
        # last layer as a quantum circuit
        #self.quantum_layer = qlayer
        self.non_quantum_layer = nn.Linear(in_features=n_qubits, out_features=2)
        #self.fc4 = nn.Linear(in_features=n_qubits, out_features=2)


    def forward(self, X):

        X = F.relu(self.conv1(X))
        X = F.max_pool2d(X, 2)

        X = F.relu(self.conv2(X))
        X = F.max_pool2d(X, 2)

        X = F.relu(self.conv3(X))
        X = F.max_pool2d(X, 2)

        X = X.view(X.shape[0], -1)
        X = F.relu(self.fc1(X))
        X = F.relu(self.fc2(X))
        X = self.fc3(X)
        # X = self.quantum_layer(X)
        X = self.non_quantum_layer(X)
        #X = self.fc4(X)

        return X

In [ ]:
def min_max_normalize(tensor):
    min_val = torch.min(tensor)
    max_val = torch.max(tensor)
    return (tensor - min_val) / (max_val - min_val)

img0 = dataset[0][0]
img1 = min_max_normalize(dataset[0][0])
plt.imshow(img1.permute(1, 2, 0))

def get_image_augm(img, datagen, count_trans_image = 1):
    fig, axes = plt.subplots(1, count_trans_image + 1, figsize=(20,10))
    image_list = [img]
    for i in range(0,count_trans_image):
        image_list.append(datagen(img))
    for i, image in enumerate(image_list):
        axes[i].imshow(image.permute(1, 2, 0), cmap='grey')
        axes[i].axis('off')

def gauss_noise_tensor(img,sigma=0.1):
    assert isinstance(img, torch.Tensor)
    dtype = img.dtype
    if not img.is_floating_point():
        img = img.to(torch.float32)
    
    out = img + sigma * torch.randn_like(img)
    
    if out.dtype != dtype:
        out = out.to(dtype)
        
    return min_max_normalize(out)

In [ ]:
# train settings
qubits_list = [2] # num of qubits
n_layers_list = [6,8,10,12] # num of entangling layers
n_base_chain_list = [1,2,3,4] # base chain * n
augment = 'No' # Yes - with distorsion, No - otherwise
iterss = 10 # Count of independment runs
now = datetime.datetime.now().strftime("%d%m%Y_%H%M%S")
epochs = 10
is_kfold = False # True / False

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(train_model.parameters(), lr = 0.001)

quant_file_name = f'quant_iters{iterss}_{"augm" if augment == "Yes" else "no_augm"}_only{''.join(map(str, qubits_list))}qubits_{now}.csv' 
classic_file_name = f'classic_iters{iterss}_{"augm" if augment == "Yes" else "no_augm"}_only{''.join(map(str, qubits_list))}finalparams_{now}.csv'

augm_dict = [transforms.ElasticTransform(alpha= 50.0),
             transforms.ElasticTransform(alpha= 75.0),
             transforms.ElasticTransform(alpha= 100.0),
             transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 5.)),
             transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)),
             transforms.GaussianBlur(kernel_size=(7, 11), sigma=(0.1, 5.)),
             transforms.Lambda(lambda x: gauss_noise_tensor(x,0.05)),
             transforms.Lambda(lambda x: gauss_noise_tensor(x,0.1)),
             transforms.Lambda(lambda x: gauss_noise_tensor(x,0.15))
            ]
df_results_class = pd.DataFrame(columns=['iteration','kfold','epoch','augment','n_qubits', 'n_layers','n_base_chain','datatype'
                                             ,'loss','accuracy', 'precision','recall', 'f1_score'
                                             ,'train_time'])
mode_model = 'quant' # 'classic' or 'quant'
for i in range(iterss):
    print(f'\n\n\n Итерация {i} \n\n\n')
    if mode_model == 'quant':
        for n_qubits in qubits_list:
            for n_layers in n_layers_list:
                for n_base_chain in n_base_chain_list:
                    print(f'Qubits {n_qubits}, entangling layers {n_layers}, base chain * {n_base_chain}')
                    train_calc(mode_model,i,str(augment),n_qubits, n_layers,n_base_chain,df_results_class,epochs,is_kfold)
                    print(f'Consumed time {time.time() - start}')
        df_results_class.to_csv(quant_file_name, sep=',', index=False, encoding='utf-8') 
    elif mode_model == 'classic':         
        for n_qubits in qubits_list: #param_coombinations.keys():
                    n_layers = None
                    n_base_chain = None
                    print(f'Полносвязный слой вместо кванотового с размерностью {n_qubits}')
                    ttrain_calc(mode_model,i,str(augment),n_qubits, n_layers,n_base_chain,df_results_class,epochs,is_kfold)
                    print(f'Затраченное время {time.time() - start}')
        df_results_class.to_csv(classic_file_name, sep=',', index=False, encoding='utf-8') 
    else:
        raise 'Incorrect mode'